In [5]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from SKBGame import SKBGame
import random

seed = 1234

class SKBEnv(gym.Env):
    def __init__(self, skb_data):
        super(SKBEnv, self).__init__()
        
        # Initialize game
        self.game = SKBGame(skb_data)
        self._data = skb_data

        # Define action space (number of available moves)
        self.action_space = spaces.Discrete(len(self.game.choices_all))

        # Define observation space (state representation)
        self.observation_space = spaces.Box(low=0, high=4, shape=(self.game.map.size.bounds[0], self.game.map.size.bounds[1]), dtype=np.float32)

        self.steps = 0
        
    def reset(self, seed=None, options=None):
        """Resets the game state."""
        super().reset(seed=seed)
        self.game = SKBGame(self._data)
        return np.array(self._get_observation(), dtype=np.float32), {}

    def step(self, action):
        """Executes an action and returns state, reward, done flag."""
        self.steps += 1
        new_game, valid = self.game.pick_all(action)
        self.game = new_game

        # Compute reward
        reward = -sum(self.game.heuristics.close) * 200
        reward -= self.steps
        
        if not valid:
            reward += -20000
            
        win_reward = 100 if self.game.win() == 1 else -300 if self.game.win() == 2 else -50
        reward += win_reward
        
        # Check if game is finished
        done = self.game.win() in [1, 2]

        return np.array(self._get_observation(), dtype=np.float32), reward, done, False, {}

    def _get_observation(self):
        """Encodes the current game state as a numerical observation."""
        char_map = {'_': 0, 'W': 1, 'B': 2, 'S': 3, 'P':4}
        return np.array([[char_map[piece] for piece in row] for row in self.game.map._skbm_grid], dtype=np.int32)

    def render(self):
        """Displays the current game state."""
        self.game.display()

env = SKBEnv('sokoban05a.txt')

In [ ]:
from stable_baselines3 import DQN
from tqdm import tqdm
import torch

model = DQN("MlpPolicy", env, verbose=0, device='cuda', batch_size=256, learning_rate=3e-4)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True

total_steps = 30
with tqdm(total=total_steps) as pbar:
    for step in range(total_steps):
        model.learn(total_timesteps=100)
        pbar.update(1)
#model.save("sokoban_ppo")


100%|██████████| 30/30 [00:09<00:00,  3.20it/s]


In [9]:
max_attempts = 100
attempts = 0

while attempts < max_attempts:
    obs, _ = env.reset()
    done = False
    while not done:
        action, _ = model.predict(obs)
        obs, reward, done, _, _ = env.step(action)
    #env.render()  # Display AI's moves
    if env.game.win() == 1:
        break
    attempts += 1
env.render()
print(f"AI completed after {attempts} attempts")

############
#   #      #
#   # #    #
#          #
###        #
#     ######
#     @$$###
############
AI completed after 3 attempts
